In [9]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

In [56]:
from typing import List
import torch
import torch.nn as nn


class Saw(nn.Module):
    def forward(self, x):
        return torch.abs(torch.remainder(x, 1.0) - 0.5) - 0.25


class MLP(nn.Module):
    def __init__(self, seq: List[int | nn.Module]):
        super().__init__()

        layers = []
        dim = seq[0]

        for item in seq[1:]:
            if isinstance(item, int):
                assert isinstance(dim, int)
                layers.append(nn.Linear(dim, item))
                dim = item
            elif isinstance(item, nn.Module):
                layers.append(item)
            else:
                raise ValueError(f"Invalid item in sequence: {item}")

        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.5)
            nn.init.normal_(m.bias, 0.0, 0.5)

    def forward(self, x):
        return self.net(x)


class MountainNoise(nn.Module):
    def __init__(self, octaves=4):
        super().__init__()
        self.octaves = nn.ModuleList([MLP([2, 32, Saw(), 1]) for _ in range(octaves)])

    def forward(self, x):
        # evaluate height
        z = 1 - abs(sum(f(x) for f in self.octaves) / len(self.octaves))

        # make it a single mountain at 0,0
        return z / (1 + torch.norm(x, dim=-1, keepdim=True) ** 2)


model = MountainNoise()
model.eval()

MountainNoise(
  (octaves): ModuleList(
    (0-3): 4 x MLP(
      (net): Sequential(
        (0): Linear(in_features=2, out_features=32, bias=True)
        (1): Saw()
        (2): Linear(in_features=32, out_features=1, bias=True)
      )
    )
  )
)

In [57]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def evaluate_on_grid(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x = np.linspace(x_range[0], x_range[1], resolution)
    y = np.linspace(y_range[0], y_range[1], resolution)
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx.flatten(), yy.flatten()], axis=-1)

    with torch.no_grad():
        inputs = torch.from_numpy(grid).float()
        raw = model(inputs).detach().cpu().numpy()
        if raw.ndim == 2 and raw.shape[1] == 1:
            raw = raw[:, 0]
        elif raw.ndim != 1:
            raise ValueError(
                f"Model must return one value per grid point, got shape {raw.shape}"
            )
        outputs = raw.reshape(resolution, resolution)

    return x, y, xx, yy, outputs


def plot_2d_3d(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x, y, xx, yy, outputs = evaluate_on_grid(
        model, x_range=x_range, y_range=y_range, resolution=resolution
    )

    # 2D heatmap and 3D surface side-by-side
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "heatmap"}, {"type": "surface"}]],
        subplot_titles=("2D Heatmap", "3D Surface"),
        horizontal_spacing=0.08,
    )

    fig.add_trace(
        go.Heatmap(
            z=outputs, x=x, y=y, colorscale="Viridis", colorbar=dict(title="output")
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Surface(z=outputs, x=xx, y=yy, colorscale="Viridis", showscale=False),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_scenes(
        xaxis_title="x", yaxis_title="y", zaxis_title="output", row=1, col=2
    )
    fig.update_layout(
        height=520, width=1200, title=f"Output over {x_range} x {y_range}"
    )
    fig.show()


s = 1
r = (-s, s)
plot_2d_3d(model, x_range=r, y_range=r, resolution=100)